### LOAD PIPELINE

In [1]:
import torch
from diffusers import StableDiffusion3Pipeline

DEVICE = "cuda"
DTYPE = torch.float16
MODEL_ID = "stabilityai/stable-diffusion-3-medium-diffusers"

pipe = StableDiffusion3Pipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
).to(DEVICE)

/home/haeun/miniconda3/envs/C3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading pipeline components...: 100%|██████████| 9/9 [00:01<00:00,  5.43it/s]


In [2]:
prompt = ""

In [4]:
import sys
sys.path.insert(0, "../")  # sd3_hook.py가 있는 경로

from sd3_hook import patch_transformer, attach_step_tracker

# 1. on_capture 콜백 정의
#    block_idx  : int
#    L_minus_1  : (batch, 4096, 1536) float16, CPU
#    L_after_attn: (batch, 4096, 1536) float16, CPU
#    L_output   : (batch, 4096, 1536) float16, CPU
#    batch_size : int (CFG면 2)
captured = {}

def on_capture(block_idx, L_minus_1, L_after_attn, L_output, batch_size):
    # .float() 먼저, 뺄셈은 나중에 (주석의 f32 변환 규칙)
    captured[block_idx] = {
        "L_minus_1":    L_minus_1.float(),
        "L_after_attn": L_after_attn.float(),
        "L_output":     L_output.float(),
    }

# 2. 블록 전체 패치 (24개 JointTransformerBlock)
already = patch_transformer(pipe.transformer, on_capture)

# 3. step tracker 부착 — 캡처할 step 번호 결정
CAPTURE_STEPS = {0, 5, 10}  # 원하는 step
step_counter = attach_step_tracker(
    pipe.transformer,
    on_step=lambda step_idx: step_idx in CAPTURE_STEPS
)

# 4. 이후 pipe() 호출하면 자동으로 캡처됨
image = pipe(prompt, num_inference_steps=28, guidance_scale=7.0).images[0]

# 5. delta 계산
block_idx = 0
d = captured[block_idx]
delta_attn = d["L_after_attn"] - d["L_minus_1"]   # attention 기여
delta_mlp  = d["L_output"]     - d["L_after_attn"] # MLP 기여


100%|██████████| 28/28 [00:50<00:00,  1.82s/it]


In [ ]:
len(captured)

3

In [8]:
len(d['L_minus_1'][0])

4096